In [ ]:
import ee
import geemap
import json
import os

import rasterio
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv

load_dotenv()  # carrega variáveis de ambiente do arquivo .env, se existir

In [ ]:
# Autenticação e inicialização do Earth Engine
# ee.Authenticate()  # colar no terminal

EE_PROJECT = os.environ.get('EE_PROJECT')
if not EE_PROJECT:
    raise RuntimeError(
        "Defina a variável de ambiente EE_PROJECT com o ID do seu projeto no Google Cloud "
        "antes de rodar esta célula (ex.: PowerShell: $env:EE_PROJECT = 'seu-projeto-id'; "
        "ou crie um arquivo .env com EE_PROJECT=seu-projeto-id)."
    )

ee.Initialize(project=EE_PROJECT)

In [ ]:
def mask_s2_clouds(image):
    """Masks clouds in a Sentinel-2 image using the QA band.

    Args:
        image (ee.Image): A Sentinel-2 image.

    Returns:
        ee.Image: A cloud-masked Sentinel-2 image.
    """
    qa = image.select('QA60')

    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    # Both flags should be set to zero, indicating clear conditions.
    mask = (
        qa.bitwiseAnd(cloud_bit_mask)
        .eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    return image.updateMask(mask).divide(10000)

In [ ]:
def extract_datacenter_timeseries(name_datacenter, lat, lon, year_list, month_start='05-01', month_end='07-30', cloud_pct=5, buffer_m=3000, scale=10, bands=['B2','B3','B4','B8','B11','B12']):
    os.makedirs('imanges_satelite', exist_ok=True)

    for year in year_list:
        dataset = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterDate(f'{year}-{month_start}', f'{year}-{month_end}')
            # Pre-filter to get less cloudy granules.
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_pct))
            .map(mask_s2_clouds)
            )

        visualization = {
            'min': 0.0,
            'max': 0.3,
            'bands':  ['B4', 'B3', 'B2'],
        }

        m = geemap.Map()
        m.set_center(lon, lat, 12)
        m.add_layer(dataset.mean(), visualization, 'RGB')

        # Cria um quadrado de ~5km ao redor do ponto central usado no mapa
        center_point = ee.Geometry.Point([lon, lat])
        region = center_point.buffer(buffer_m).bounds()

        # --- Preparar a imagem final para exportação ---
        image_to_export = dataset.mean().select(bands)

        # --- Baixar direto para a máquina local ---
        geemap.ee_export_image(
            image_to_export,
            filename=f'imanges_satelite/{name_datacenter}_{year}.tif',
            scale=scale,
            region=region,
            file_per_band=False
        )

        print(f'[{name_datacenter} {year}] Download concluído.')

    # --- Salva os metadados usados, pra reaproveitar depois na classificação ---
    metadata = {
        'name_datacenter': name_datacenter,
        'lat': lat,
        'lon': lon,
        'year_list': year_list,
        'bands': bands,
        'buffer_m': buffer_m,
        'scale': scale,
        'crs': 'EPSG:4326',
    }
    with open(f'imanges_satelite/{name_datacenter}_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f'\nMetadados salvos em imanges_satelite/{name_datacenter}_metadata.json')

In [ ]:
name_datacenter = 'Ascenty_Vinhedo'
lat = -23.071035
lon = -47.011837
year_list = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

extract_datacenter_timeseries(name_datacenter, lat, lon, year_list)

In [ ]:


path = 'imanges_satelite/Ascenty_Vinhedo_2024.tif'

with rasterio.open(path) as src:
    # Bandas na ordem que você exportou: B2,B3,B4,B8,B11,B12
    # Para RGB "natural", precisamos de B4(vermelho), B3(verde), B2(azul)
    red = src.read(3)    # B4
    green = src.read(2)  # B3
    blue = src.read(1)   # B2

rgb = np.dstack([red, green, blue])
rgb = np.clip(rgb / 0.3, 0, 1)  # normaliza pro range visível (mesmo min/max do seu 'visualization')

plt.figure(figsize=(8, 8))
plt.imshow(rgb)
plt.title('Ascenty Vinhedo 2024')
plt.axis('off')
plt.show()

In [ ]:
os.list

In [ ]:
import os
import rasterio
import matplotlib.pyplot as plt
import numpy as np

pasta_entrada = 'imanges_satelite'
pasta_saida = 'imagens_jpg'

os.makedirs(pasta_saida, exist_ok=True)  # cria a pasta se não existir

for nome_arquivo in os.listdir(pasta_entrada):
    if not nome_arquivo.endswith('.tif'):
        continue

    path = os.path.join(pasta_entrada, nome_arquivo)

    with rasterio.open(path) as src:
        # Bandas na ordem que você exportou: B2,B3,B4,B8,B11,B12
        red = src.read(3)    # B4
        green = src.read(2)  # B3
        blue = src.read(1)   # B2

    rgb = np.dstack([red, green, blue])
    rgb = np.clip(rgb / 0.3, 0, 1)

    nome_saida = os.path.splitext(nome_arquivo)[0] + '.jpg'
    path_saida = os.path.join(pasta_saida, nome_saida)

    plt.figure(figsize=(8, 8))
    plt.imshow(rgb)
    plt.title(nome_arquivo)
    plt.axis('off')
    plt.savefig(path_saida, dpi=150, bbox_inches='tight')
    plt.close()

    print(f'Salvo: {path_saida}')